# Smoke test — traceable comparables from a *local* TabPFN-3.5

**Purpose.** Decide, before any repository work, whether TabPFN-3.5 can hand back, for each appraised
property, the training transactions that support its estimate ("comparables"), in a form a forensic
appraiser can audit line by line.

**What is known before running** (from the `tabpfn` 9.0 / `tabpfn-extensions` 0.6 source, 2026-09-28):

* The exact per-training-row readout (`get_decoder_readout`) reads the attention weights of the
  `ManyClassDecoder`. **Only the classification head has this decoder.** The regression head is a linear
  layer over distribution buckets — there is no exact row-vote decomposition of a `TabPFNRegressor` prediction.
* Both readout and embeddings need the **local** model (the hosted API does not expose internals).
  Weights: `Prior-Labs/tabpfn_3_5` (research/eval licence, accepted at https://ux.priorlabs.ai → *License*),
  fetched with the same `TABPFN_TOKEN` used for the API. **No API credits are spent here.**

**Two routes are therefore tested, with pre-registered pass/fail criteria:**

| Route | Object | What it gives | Criterion |
|---|---|---|---|
| **A** | local `TabPFNRegressor` (the submitted model, same T0 features) | (A1) same predictions as the API fit; (A2) nearest training rows in the model's *embedding* space — a similarity ranking, not weights | A1: Pearson(ŷ_local, ŷ_API) ≥ 0.98 and mean \|Δ\| ≤ 0.05 ln at full 24 k. A2: comparables are geographically closer than random rows (median top-20 distance < ½ of a random row's) |
| **B** | companion `TabPFNClassifier` on `ln_vu` cut into K quantile bins | exact non-negative weights over training rows summing to 1; readout estimate ŷ_B = Σ wⱼ·ln_vuⱼ | B1: Σw = 1 and class-vote reproduces `predict_proba` (max abs diff < 0.02). **B2 (decisive):** Pearson(ŷ_B, ŷ_local regressor) ≥ 0.90 and RMSE(ŷ_B − ŷ_local) ≤ 0.15 ln. B3: weights decay with distance (median Spearman(w, dist) < −0.10) |

Reading of the outcome: if **B2 fails**, the companion classifier's comparables do *not* explain the
regressor's estimate and only Route A (similarity, no weights) survives; if **A1 fails**, the local weights
are not the model that was evaluated and the whole line stops. Everything below is on the 50-property
2026 subsample already used for SHAP (`data/shap_subsample.csv`), so the results can sit next to the
SHAP figures.

> The weights explain the model's computation. They do not establish that a training transaction
> *caused* the estimate — in a report: "elements that supported the estimate".

In [ ]:
# --- 0. Environment ------------------------------------------------------------
import subprocess, sys, os, time, shutil
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print(gpu or "NO GPU — Runtime > Change runtime type > T4 GPU (the 24k fits need CUDA)")

In [ ]:
%pip install -q -U "tabpfn>=9.0.0" "tabpfn-extensions>=0.6.3" umap-learn
import tabpfn, tabpfn_extensions, torch
print("tabpfn", tabpfn.__version__, "| tabpfn-extensions", tabpfn_extensions.__version__,
      "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Prior Labs token (same one as the API). Colab: key icon on the left → add secret TABPFN_TOKEN.
# The licence for the TabPFN-3.5 weights must have been accepted once at https://ux.priorlabs.ai (License tab).
try:
    from google.colab import userdata
    os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
except Exception as e:
    print("not on Colab or secret missing:", e)
assert os.environ.get("TABPFN_TOKEN"), "TABPFN_TOKEN not set"
print("token set:", os.environ["TABPFN_TOKEN"][:4] + "…")

## 1. Data — straight from the public repository

In [ ]:
import numpy as np, pandas as pd
RAW = "https://raw.githubusercontent.com/abenvenho/tabpfn-itbi-sp/main/"
DTYPES = {"sql": str, "cep": str, "setor": str, "setor_quadra": str, "padrao_iptu": str, "padrao_tipo": str}

level_a = pd.read_csv(RAW + "data/itbi_sp_2025_level_a.csv", dtype=DTYPES)          # 24 000 training rows (2025)
test26  = pd.read_csv(RAW + "data/itbi_sp_2026_test.csv",     dtype=DTYPES)          # 47 810 rows (2026)
shap_sub = pd.read_csv(RAW + "data/shap_subsample.csv", dtype={"sql": str})          # the 50 SHAP properties
oot_pred = pd.read_csv(RAW + "results/oot_pred_tabpfn_t0_level_a_seed42.csv", dtype={"sql": str})  # API predictions, 24k fit

assert (oot_pred["sql"].values == test26["sql"].values).all(), "oot_pred is row-aligned with test26"
test26["yhat_api"] = oot_pred["yhat_ln"].values

# the 50 queries: exact match on (sql, date, price) recovers their test rows (and the API prediction)
key = ["sql", "data_transacao", "valor_transacao"]
queries = test26.merge(shap_sub[key], on=key, how="inner").reset_index(drop=True)
assert len(queries) == 50, len(queries)
print(level_a.shape, test26.shape, queries.shape)

In [ ]:
# T0 feature table — replica of src/model_tabpfn.py::t0_features (raw columns, no spatial machinery)
T0_NUMERIC = ["area_construida_m2", "area_terreno_m2", "idade", "padrao_nivel", "dist_estacao_m", "mes_idx", "lat", "lon"]
T0_CATEGORICAL = ["tipo_imovel"]
TARGET = "ln_vu"

def t0_features(df):
    X = pd.DataFrame(index=df.index)
    for c in T0_NUMERIC:
        X[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    X["area_terreno_m2"] = X["area_terreno_m2"].fillna(0.0)
    for c in T0_CATEGORICAL:
        X[c] = df[c].astype("category")
    return X.reset_index(drop=True)

# N_TRAIN: 4000 for a first pass (≈1 min), then 24000 (the fit that was actually evaluated).
# Criterion A1 is only meaningful at 24000.
N_TRAIN = 24000
rng = np.random.default_rng(42)
train = level_a if N_TRAIN >= len(level_a) else level_a.iloc[np.sort(rng.choice(len(level_a), N_TRAIN, replace=False))]
train = train.reset_index(drop=True)

X_train, y_train = t0_features(train), train[TARGET].to_numpy(float)
X_q, y_q = t0_features(queries), queries[TARGET].to_numpy(float)
cat_idx = [X_train.columns.get_loc(c) for c in T0_CATEGORICAL]
print(f"train {X_train.shape} | queries {X_q.shape} | categorical idx {cat_idx}")

In [ ]:
# geographic distances query → every training row (haversine, metres) — used by A2 and B3
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dlmb = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

D = haversine_m(queries["lat"].to_numpy()[:, None], queries["lon"].to_numpy()[:, None],
                train["lat"].to_numpy()[None, :], train["lon"].to_numpy()[None, :])   # (50, n_train)
print("median distance query→random training row: %.0f m" % np.median(D))

## 2. Route A — the regressor itself, locally

A1 checks that the local checkpoint reproduces the API fit that was evaluated in the out-of-time leg
(`results/oot_pred_tabpfn_t0_level_a_seed42.csv`). A2 uses the model's own row embeddings
(`get_embeddings`) to rank training rows by similarity — the closest thing to "comparables as the model
sees them" that exists for the regression head.

In [ ]:
from tabpfn import TabPFNRegressor
t0 = time.time()
reg = TabPFNRegressor.create_default_for_version("v3.5", device="cuda", random_state=42,
                                                  categorical_features_indices=cat_idx,
                                                  ignore_pretraining_limits=True)
reg.fit(X_train, y_train)
yhat_local = reg.predict(X_q)
print(f"fit+predict {time.time()-t0:.0f}s")

yhat_api = queries["yhat_api"].to_numpy(float)
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
A1 = {"pearson_local_vs_api": float(np.corrcoef(yhat_local, yhat_api)[0, 1]),
      "mean_abs_diff_local_vs_api": float(np.mean(np.abs(yhat_local - yhat_api))),
      "rmse_local_vs_truth": rmse(yhat_local, y_q), "rmse_api_vs_truth": rmse(yhat_api, y_q),
      "n_train": int(len(X_train))}
A1["pass"] = bool(A1["n_train"] == 24000 and A1["pearson_local_vs_api"] >= 0.98 and A1["mean_abs_diff_local_vs_api"] <= 0.05)
A1

In [ ]:
# A2 — embedding-space neighbours (cosine), top-20 per query
emb_train = reg.get_embeddings(X_q, data_source="train").mean(axis=0)   # (n_train, d), averaged over estimators
emb_q     = reg.get_embeddings(X_q, data_source="test").mean(axis=0)    # (50, d)
def _norm(a): return a / np.linalg.norm(a, axis=1, keepdims=True)
S = _norm(emb_q) @ _norm(emb_train).T                                    # (50, n_train) cosine similarity
TOP = 20
topA = np.argsort(-S, axis=1)[:, :TOP]

d_topA = np.take_along_axis(D, topA, axis=1)                             # distances of the top-20 similar rows
d_rand = D[:, rng.choice(D.shape[1], TOP, replace=False)]
A2 = {"median_dist_top20_m": float(np.median(d_topA)), "median_dist_random20_m": float(np.median(d_rand)),
      "share_top20_within_1km": float((d_topA < 1000).mean()),
      "share_top20_same_type": float(np.mean(train["tipo_imovel"].to_numpy()[topA] == queries["tipo_imovel"].to_numpy()[:, None]))}
# similarity-weighted mean of the neighbours' ln_vu vs the regressor's own prediction (softmax over cosine)
w_soft = np.exp(20 * np.take_along_axis(S, topA, axis=1)); w_soft /= w_soft.sum(1, keepdims=True)
yhat_A = (w_soft * y_train[topA]).sum(1)
A2["pearson_knn_estimate_vs_local"] = float(np.corrcoef(yhat_A, yhat_local)[0, 1])
A2["pass"] = bool(A2["median_dist_top20_m"] < 0.5 * A2["median_dist_random20_m"])
A2

## 3. Route B — companion classifier with the exact decoder readout

`ln_vu` is cut into K quantile bins; the classifier is fitted on the same T0 features with
`softmax_temperature=1.0`, `balance_probabilities=False`, float32 (the settings under which the class vote
reproduces `predict_proba`). The readout gives one weight per (query, training row). The **readout
estimate** Σ wⱼ·ln_vuⱼ uses the *actual* continuous targets of the training rows — it is a weighted
average of real transactions, exactly the arithmetic of the comparative method.

In [ ]:
from tabpfn import TabPFNClassifier
from tabpfn_extensions.interpretability import get_decoder_readout, class_vote

K = 10
bins = pd.qcut(y_train, q=K, labels=False, duplicates="drop").astype(int)
print("bins:", np.bincount(bins))

t0 = time.time()
clf = TabPFNClassifier.create_default_for_version("v3.5", device="cuda", random_state=42,
                                                   categorical_features_indices=cat_idx,
                                                   softmax_temperature=1.0, balance_probabilities=False,
                                                   inference_precision=torch.float32,
                                                   ignore_pretraining_limits=True)
clf.fit(X_train, bins)
proba = clf.predict_proba(X_q)
weights, train_idx = get_decoder_readout(clf, X_q)        # (50, n_train), rows sum to 1
print(f"fit+readout {time.time()-t0:.0f}s | weights {weights.shape}")

votes, classes = class_vote(weights, bins[train_idx])
B1 = {"row_sums_min_max": [float(weights.sum(1).min()), float(weights.sum(1).max())],
      "min_weight": float(weights.min()),
      "max_abs_diff_vote_vs_proba": float(np.abs(votes - proba).max())}
B1["pass"] = bool(abs(B1["row_sums_min_max"][0] - 1) < 1e-3 and abs(B1["row_sums_min_max"][1] - 1) < 1e-3
                  and B1["min_weight"] >= -1e-8 and B1["max_abs_diff_vote_vs_proba"] < 0.02)
B1

In [ ]:
# B2 — does the companion's weighted average of real transactions track the regressor?
y_train_al = y_train[train_idx]
yhat_B = weights @ y_train_al
B2 = {"pearson_readout_vs_local_reg": float(np.corrcoef(yhat_B, yhat_local)[0, 1]),
      "rmse_readout_vs_local_reg": rmse(yhat_B, yhat_local),
      "pearson_readout_vs_api": float(np.corrcoef(yhat_B, yhat_api)[0, 1]),
      "rmse_readout_vs_truth": rmse(yhat_B, y_q), "rmse_local_reg_vs_truth": rmse(yhat_local, y_q),
      "mean_bias_readout_minus_local": float(np.mean(yhat_B - yhat_local))}
B2["pass"] = bool(B2["pearson_readout_vs_local_reg"] >= 0.90 and B2["rmse_readout_vs_local_reg"] <= 0.15)
B2

In [ ]:
# B3 — concentration and spatial behaviour of the weights, per query
from scipy.stats import spearmanr
D_al = D[:, train_idx]
order = np.argsort(-weights, axis=1)
w_sorted = np.take_along_axis(weights, order, axis=1)
cum = np.cumsum(w_sorted, axis=1)
per_q = pd.DataFrame({
    "sql": queries["sql"], "tipo_imovel": queries["tipo_imovel"], "bloco": queries["bloco"],
    "y_ln": y_q, "yhat_local": yhat_local, "yhat_api": yhat_api, "yhat_readout": yhat_B,
    "n_eff": 1.0 / (weights ** 2).sum(1),
    "rows_for_50pct": (cum < 0.5).sum(1) + 1, "rows_for_90pct": (cum < 0.9).sum(1) + 1,
    "top20_mass": w_sorted[:, :TOP].sum(1),
    "top20_median_dist_m": np.median(np.take_along_axis(D_al, order[:, :TOP], axis=1), axis=1),
    "top20_share_same_type": np.mean(train["tipo_imovel"].to_numpy()[train_idx][order[:, :TOP]]
                                     == queries["tipo_imovel"].to_numpy()[:, None], axis=1),
    "weighted_mean_dist_m": (weights * D_al).sum(1),
    "spearman_w_dist": [spearmanr(weights[i], D_al[i]).statistic for i in range(len(queries))],
})
per_q["abs_err_local"] = np.abs(per_q["yhat_local"] - per_q["y_ln"])
B3 = {"median_spearman_w_dist": float(per_q["spearman_w_dist"].median()),
      "median_top20_dist_m": float(per_q["top20_median_dist_m"].median()),
      "median_n_eff": float(per_q["n_eff"].median()),
      "median_rows_for_90pct": float(per_q["rows_for_90pct"].median()),
      "spearman_n_eff_vs_abs_err": float(spearmanr(per_q["n_eff"], per_q["abs_err_local"]).statistic),
      "overlap_top20_routeA_routeB": float(np.mean([len(set(topA[i]) & set(train_idx[order[i, :TOP]])) / TOP for i in range(len(queries))]))}
B3["pass"] = bool(B3["median_spearman_w_dist"] < -0.10)
display(per_q.describe().T.round(3)); B3

## 4. Two pictures — one appraisal as a table and on the map

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")
# the query whose readout estimate is most uncertain in the sense of a flat vote: lowest top-20 mass
q = int(np.argmin(per_q["top20_mass"]))
tr = train.iloc[train_idx].reset_index(drop=True)
tab = tr.iloc[order[q, :12]][["sql", "tipo_imovel", "descricao_padrao", "area_construida_m2", "idade", "mes_referencia", "ln_vu"]].copy()
tab.insert(0, "weight", weights[q, order[q, :12]]); tab["dist_m"] = D_al[q, order[q, :12]].round(0)
tab["VU_R$/m2"] = np.exp(tab["ln_vu"]).round(0)
print(f"query {queries.loc[q,'sql']} ({queries.loc[q,'tipo_imovel']}, block {queries.loc[q,'bloco']}): "
      f"truth {y_q[q]:.3f} | regressor {yhat_local[q]:.3f} | readout {yhat_B[q]:.3f} | n_eff {per_q.loc[q,'n_eff']:.0f}")
display(tab.style.format({"weight": "{:.2%}"}).bar(subset=["weight"], color="#8CB9F1"))

In [ ]:
# map: 4 queries (lowest / highest n_eff, worst / best local error), lines to their top-20 comparables scaled by weight
picks = [int(per_q["n_eff"].idxmin()), int(per_q["n_eff"].idxmax()),
         int(per_q["abs_err_local"].idxmax()), int(per_q["abs_err_local"].idxmin())]
titles = ["most concentrated (min n_eff)", "most diffuse (max n_eff)", "largest |error|", "smallest |error|"]
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, qi, t in zip(axes.ravel(), picks, titles):
    ax.scatter(tr["lon"], tr["lat"], s=2, c="#cccccc", alpha=0.4, linewidths=0)
    top = order[qi, :TOP]; wmax = weights[qi, top].max()
    for j in top:
        f = weights[qi, j] / wmax
        ax.plot([queries.loc[qi, "lon"], tr.loc[j, "lon"]], [queries.loc[qi, "lat"], tr.loc[j, "lat"]],
                color="#2a78d6", lw=0.6 + 3 * f, alpha=0.3 + 0.6 * f)
    sc = ax.scatter(tr.loc[top, "lon"], tr.loc[top, "lat"], s=40 + 400 * weights[qi, top] / wmax,
                    c=tr.loc[top, "ln_vu"], cmap="viridis", edgecolor="white", zorder=3)
    ax.scatter(queries.loc[qi, "lon"], queries.loc[qi, "lat"], marker="*", s=500, c="#eb6834", edgecolor="k", zorder=4)
    # window: fit the top-20 (at least ±500 m, at most ±10 km); count the ones left outside
    off = np.maximum(np.abs(tr.loc[top, "lon"] - queries.loc[qi, "lon"]), np.abs(tr.loc[top, "lat"] - queries.loc[qi, "lat"]))
    pad = float(np.clip(1.15 * off.max(), 0.005, 0.10)); outside = int((off > pad).sum())
    ax.set_xlim(queries.loc[qi, "lon"] - pad, queries.loc[qi, "lon"] + pad); ax.set_ylim(queries.loc[qi, "lat"] - pad, queries.loc[qi, "lat"] + pad)
    ax.set_title(f"{t} — window ±{pad*111:.1f} km" + (f", {outside} of 20 beyond it" if outside else "") +
                 f"\n{queries.loc[qi,'tipo_imovel']} · truth {y_q[qi]:.2f} · regressor {yhat_local[qi]:.2f} · readout {yhat_B[qi]:.2f} · "
                 f"top-20 hold {per_q.loc[qi,'top20_mass']:.0%}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect(1 / np.cos(np.radians(-23.55)))
fig.colorbar(sc, ax=axes, shrink=0.5, label="ln VU of comparable")
fig.suptitle("Decoder-readout comparables (companion classifier): star = appraised property, "
             "line width = weight", fontsize=13, fontweight="bold")
plt.show()

## 5. Verdict — written to `out/`

In [ ]:
import json, pathlib
out = pathlib.Path("out"); out.mkdir(exist_ok=True)
verdict = {"n_train": int(len(X_train)), "K_bins": int(K), "tabpfn": tabpfn.__version__,
           "tabpfn_extensions": tabpfn_extensions.__version__,
           "A1_local_reproduces_api": A1, "A2_embedding_neighbours_are_local": A2,
           "B1_readout_is_exact": B1, "B2_readout_tracks_regressor": B2, "B3_weights_decay_with_distance": B3}
for k, v in verdict.items():
    if isinstance(v, dict) and "pass" in v:
        print(f"{'PASS' if v['pass'] else 'FAIL'}  {k}")
(out / "smoke_comparables_local.json").write_text(json.dumps(verdict, indent=2))
per_q.to_csv(out / "smoke_comparables_per_query.csv", index=False)
fig.savefig(out / "smoke_comparables_map.png", dpi=150, bbox_inches="tight")
np.save(out / "smoke_readout_weights.npy", weights.astype(np.float32))   # (50, n_train) — for the repo analysis later
print("\nwritten:", sorted(p.name for p in out.iterdir()))

In [ ]:
# download the results (or mount Drive and copy `out/` — whichever you prefer)
import shutil
from google.colab import files
shutil.make_archive("smoke_comparables_local", "zip", "out")
files.download("smoke_comparables_local.zip")

### How to read the outcome

* **A1 fail** → the local checkpoint is not the model that was evaluated; stop this line, report why.
* **A1 pass, B2 fail** → only Route A survives: comparables can be shown as *the most similar transactions in
  the model's representation*, without weights and without an arithmetic link to the estimate. Honest, weaker.
* **A1 and B2 pass** → the companion classifier is a defensible readout of the appraisal: comparables with
  weights, effective number of elements, and a value that is literally a weighted average of real transactions.
  Next step is the repository analysis on all 50 SHAP properties (weights saved above), then the map figure and
  the README section. If B3 also passes, that is mechanistic evidence for the central hypothesis: a model given
  latitude and longitude as two plain columns concentrates its vote on geographic neighbours.